## RAG Ingestion 3 - Incremental: only changed content goes to Pinecone
Each chunk's id is built from a hash of its text: `<file>#<sha256(chunk)>`.
The same text always gets the same id, so on every run we compare:

- **new ids** (in the file, not in Pinecone) -> embed and upsert only these
- **old ids** (in Pinecone, not in the file anymore) -> delete them, so outdated content can't be retrieved
- **ids in both** -> unchanged, nothing to embed

An identical file therefore costs no embeddings and no writes, and an edited file
only re-embeds the chunks whose text changed.

### Step 1: Imports and configuration

In [50]:
import os
import re
import time
import hashlib
from dotenv import load_dotenv
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from sentence_transformers import SentenceTransformer
from pinecone import Pinecone, ServerlessSpec

load_dotenv()

DATA_FOLDER = "Leave_Policy"
NAMESPACE = "rag-ingestion-incremental"   # own namespace so it doesn't mix with other notebooks' data
SUPPORTED = (".pdf", ".md", ".txt")

### Step 2: Connect to Pinecone (create the index if it doesn't exist)

In [51]:
pc = Pinecone(api_key=os.getenv("PineCone_API_Key"))
index_name = os.getenv("Pinecone_Index_Name")

if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
index = pc.Index(index_name)
print(index.describe_index_stats())

DescribeIndexStatsResponse(dimension=384, total_vector_count=0, metric='cosine', namespaces=0)


### Step 3: Read files
List the supported files in the folder and read their text (PDFs page by page).

In [52]:
def list_files(folder):
    return sorted(
        os.path.join(folder, f) for f in os.listdir(folder) if f.lower().endswith(SUPPORTED)
    )


def file_id(path):
    """Pinecone-safe id prefix built from the file name."""
    return re.sub(r"[^A-Za-z0-9_.-]", "_", os.path.basename(path))


def load_pages(path):
    """Text of each PDF page, or the whole md/txt file as a single page."""
    if path.lower().endswith(".pdf"):
        return [p.page_content for p in PyPDFLoader(path).load()]
    with open(path, encoding="utf-8") as f:
        return [f.read()]


files = list_files(DATA_FOLDER)
print("Files found:", [file_id(p) for p in files])

pages = load_pages(files[0])
print(f"\n{file_id(files[0])}: {len(pages)} pages")
print(pages[0][:300])

Files found: ['Leave_Policy_2015.pdf', 'Leave_Policy_2016.pdf', 'Leave_Policy_2017.pdf', 'Leave_Policy_2018.pdf', 'Leave_Policy_2019.pdf', 'Leave_Policy_2020.pdf', 'Leave_Policy_2021.pdf', 'Leave_Policy_2022.pdf', 'Leave_Policy_2023.pdf', 'Leave_Policy_2024.pdf', 'Leave_Policy_2025.pdf', 'Leave_Policy_2026.pdf']

Leave_Policy_2015.pdf: 3 pages
Acme Technologies India Pvt. Ltd. | Leave Policy 2015 | HR-LV-2015
Page 1
 Leave Policy 2015
 Acme Technologies India Pvt. Ltd.
Field
Details
Document ID
HR-LV-2015
Policy year
2015
Effective period
1 January 2015 to 31 December 2015
Supersedes
None (first consolidated policy)
Owner
Human Resources 


### Step 4: Chunking
Split each page into chunks, remembering the page number.

Split on headings first and use no overlap, so an edit only changes the chunks of its own section instead of shifting every chunk after it.

In [53]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=150,
    chunk_overlap=0,
    separators=["\n## ", "\n\n", "\n", " ", ""],
)


def chunk_file(path):
    """List of (chunk_text, page_no) for one file."""
    chunks = []
    for page_no, page in enumerate(load_pages(path), start=1):
        chunks += [(text, page_no) for text in splitter.split_text(page)]
    return chunks


sample_chunks = chunk_file(files[0])
print(f"{file_id(files[0])}: {len(sample_chunks)} chunks")
for text, page_no in sample_chunks[:3]:
    print(f"\n--- page {page_no}, length {len(text)} ---")
    print(text)

Leave_Policy_2015.pdf: 56 chunks

--- page 1, length 141 ---
Acme Technologies India Pvt. Ltd. | Leave Policy 2015 | HR-LV-2015
Page 1
 Leave Policy 2015
 Acme Technologies India Pvt. Ltd.
Field
Details

--- page 1, length 141 ---
Document ID
HR-LV-2015
Policy year
2015
Effective period
1 January 2015 to 31 December 2015
Supersedes
None (first consolidated policy)
Owner

--- page 1, length 126 ---
Human Resources Department
Applies to
All permanent employees of Acme Technologies India Pvt. Ltd.
1. Purpose and Scope (2015)


### Step 5: Embedding
Convert chunk text into 384-dimension vectors.

In [54]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


def embed(texts):
    return model.encode(texts)


sample_vectors = embed([text for text, _ in sample_chunks[:3]])
print("Vector shape:", sample_vectors.shape)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 958.71it/s]


Vector shape: (3, 384)


### Step 6: Insert into Pinecone
Upsert records in batches of 100.

In [ ]:
def upsert_batches(records, batch_size=100):
    for start in range(0, len(records), batch_size):
        index.upsert(vectors=records[start:start + batch_size], namespace=NAMESPACE)
        index.search_records(vector=PROBE, top_k=1, namespace=NAMESPACE,match_terms={"source": {"*": "emp"}})

        name = "emp"
        name.__contains__("emp")

### Step 7: Chunk ids from content hash
Same text -> same id, edited text -> new id.

In [58]:
def chunk_id(fid, text):
    return f"{fid}#{hashlib.sha256(text.encode('utf-8')).hexdigest()[:16]}"


def chunk_map(path):
    """chunk id -> (text, page_no); identical chunks collapse into one."""
    fid = file_id(path)
    chunks = {}
    for text, page_no in chunk_file(path):
        chunks.setdefault(chunk_id(fid, text), (text, page_no))
    return chunks


for cid in list(chunk_map(files[0]))[:3]:
    print(cid)

Leave_Policy_2015.pdf#bca83540b2bd475c
Leave_Policy_2015.pdf#a5ad957420c73d82
Leave_Policy_2015.pdf#7e6b984ef1c9424d


### Step 8: Chunk ids already stored in Pinecone for a file

In [59]:
def existing_ids(fid):
    ids = set()
    for page in index.list(prefix=f"{fid}#", namespace=NAMESPACE):
        for item in page:
            ids.add(item.id if hasattr(item, "id") else item)
    return ids

### Step 9: Compare - which chunks to add, which to delete

In [60]:
def diff_chunks(chunks, old):
    to_add = [cid for cid in chunks if cid not in old]     
    return to_add

### Step 10: Apply the changes - embed + insert new chunks, delete removed ones

In [61]:
def add_chunks(fid, chunks, to_add):
    if not to_add:
        return
    vectors = embed([chunks[cid][0] for cid in to_add])
    upsert_batches([
        {
            "id": cid,
            "values": vectors[i].tolist(),
            "metadata": {"text": chunks[cid][0], "source": fid, "page": chunks[cid][1]},
        }
        for i, cid in enumerate(to_add)
    ])


### Step 11: Sync a file / folder (steps 7-10 together)

In [62]:
def sync_file(path):
    fid = file_id(path)
    chunks = chunk_map(path)
    old = existing_ids(fid)
    to_add = diff_chunks(chunks, old)

    add_chunks(fid, chunks, to_add)
    

    if not to_add :
        print(f"UNCHANGED {fid}: {len(chunks)} chunks, nothing to do")
    else:
        status = "NEW      " if not old else "UPDATED  "
        print(f"{status} {fid}: kept {len(chunks) - len(to_add)}, "
              f"added {len(to_add)}")
    return to_add


def sync_folder(folder):
    for path in list_files(folder):
        sync_file(path)

### Step 12: Sync the leave policy PDFs
First run adds everything; re-running this cell changes nothing.

In [63]:
sync_folder(DATA_FOLDER)

NEW       Leave_Policy_2015.pdf: kept 0, added 56
NEW       Leave_Policy_2016.pdf: kept 0, added 64
NEW       Leave_Policy_2017.pdf: kept 0, added 65
NEW       Leave_Policy_2018.pdf: kept 0, added 64
NEW       Leave_Policy_2019.pdf: kept 0, added 67
NEW       Leave_Policy_2020.pdf: kept 0, added 70
NEW       Leave_Policy_2021.pdf: kept 0, added 69
NEW       Leave_Policy_2022.pdf: kept 0, added 78
NEW       Leave_Policy_2023.pdf: kept 0, added 68
NEW       Leave_Policy_2024.pdf: kept 0, added 75
NEW       Leave_Policy_2025.pdf: kept 0, added 80
NEW       Leave_Policy_2026.pdf: kept 0, added 72


### Step 13: Demo - version 1 of a handbook file

In [ ]:
DEMO_FOLDER = "RAG_Incremental_Demo"
os.makedirs(DEMO_FOLDER, exist_ok=True)
demo_file = os.path.join(DEMO_FOLDER, "employee_handbook.md")

v1 = """# Employee Handbook

## Leave Policy

Employees receive 20 days of annual leave each year.
Manager approval is required before taking leave.

## Working Hours

The standard working day is from 9 AM to 6 PM.
Employees must record their attendance.

## IT Security

Employees must use multi-factor authentication.
Company passwords must never be shared.
"""
with open(demo_file, "w", encoding="utf-8") as f:
    f.write(v1)

sync_file(demo_file)

### Step 14: Same file again - nothing is embedded or written

In [ ]:
time.sleep(5)  # freshly written ids take a few seconds to show up in index.list
sync_file(demo_file)

### Step 15: Same file, some content updated
Annual leave changed 20 -> 24 days and a new section added.
Expected: kept 2, added 2 (edited Leave Policy chunk + Remote Work), deleted 1 (old "20 days" chunk).

In [ ]:
v2 = v1.replace("20 days of annual leave", "24 days of annual leave") + """
## Remote Work

Employees may work from home up to 2 days per week.
"""
with open(demo_file, "w", encoding="utf-8") as f:
    f.write(v2)

added, deleted = sync_file(demo_file)
print("Added ids:  ", added)
print("Deleted ids:", deleted)

### Step 16: Retrieval check

In [ ]:
def search(question, top_k=3):
    res = index.query(
        vector=embed(question).tolist(),
        top_k=top_k,
        include_metadata=True,
        namespace=NAMESPACE,
    )
    for m in res.matches:
        print(f"\nScore: {m.score:.3f} | {m.metadata['source']} (page {m.metadata['page']})")
        print(m.metadata["text"])


time.sleep(5)  # stats can lag briefly
search("How many days of annual leave do employees get?", top_k=2)